# London's Blue Plaques: a geospatial & optimisation playground

The companion notebook asked *who* gets remembered. This one treats the 1,035 geolocated plaques as a **spatial point pattern** and an **optimisation problem**: how clustered is London's memory (statistically), what territory does each plaque own, where are the natural hotspots, and what's the shortest walking tour of the city?

All distances use the **British National Grid (EPSG:27700)**, so lengths and areas are in real metres.

**Data:** English Heritage blue plaques, scraped July 2026 (see `scrape_plaques.py`). A personal, non-commercial analysis with attribution; full licence in the [companion post](https://koulakhilesh.github.io/writing/london-blue-plaques/).

In [21]:
import json
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.spatial import cKDTree
from scipy.spatial.distance import pdist, squareform
from shapely.geometry import MultiPoint
from shapely.ops import voronoi_diagram
from sklearn.cluster import DBSCAN

df = pd.read_csv("../data/blue_plaques_2026-07.csv")
df["borough_short"] = (df["borough"].astype(str)
    .str.replace("London Borough of ", "", regex=False)
    .str.replace("Royal Borough of ", "", regex=False)
    .str.replace("City Of ", "", regex=False).str.strip())
geo = df.dropna(subset=["lat", "lng"]).copy()
CENTER = {"lat": geo["lat"].mean(), "lon": geo["lng"].mean()}

# Project WGS84 -> British National Grid (EPSG:27700) for metric distances/areas.
pts = gpd.GeoDataFrame(geo, geometry=gpd.points_from_xy(geo["lng"], geo["lat"]), crs=4326)
pts_m = pts.to_crs(27700)
X = np.c_[pts_m.geometry.x.values, pts_m.geometry.y.values]
print(f"{len(X)} geolocated plaques projected to metres")

1035 geolocated plaques projected to metres


In [22]:
# House style: one consistent look for every chart in this notebook.
import plotly.io as pio
import plotly.graph_objects as go

pio.templates["plaque"] = go.layout.Template(layout=dict(
    font=dict(family="-apple-system, 'Segoe UI', Helvetica, Arial, sans-serif",
              size=13, color="#2b2b2b"),
    title=dict(font=dict(size=18, color="#14213d"), x=0.02, xanchor="left"),
    colorway=["#1c5fb0", "#c2185b", "#2e8b8b", "#e08a1e", "#6a4c93", "#417a3f", "#b0413e"],
    paper_bgcolor="white", plot_bgcolor="white",
    xaxis=dict(gridcolor="#eef1f5", linecolor="#d5dae1", zeroline=False,
               ticks="outside", tickcolor="#d5dae1", ticklen=4),
    yaxis=dict(gridcolor="#eef1f5", linecolor="#d5dae1", zeroline=False),
    margin=dict(t=62, r=26, b=48, l=26),
    hoverlabel=dict(bgcolor="white", font=dict(size=12)),
))
pio.templates.default = "plotly_white+plaque"
px.defaults.template = "plotly_white+plaque"

## How clustered is it, really? (Clark-Evans)

Before any picture, a statistic. The **Clark-Evans index** compares the mean nearest-neighbour distance to what you'd expect if the plaques were scattered at random. `R < 1` means clustered; a large negative `z` makes it statistically undeniable.

In [23]:
tree = cKDTree(X)
nn = tree.query(X, k=2)[0][:, 1]                 # nearest-neighbour distance, metres
hull_m = MultiPoint(list(map(tuple, X))).convex_hull
n = len(X); density = n / hull_m.area
r_exp = 0.5 / np.sqrt(density)
R = nn.mean() / r_exp
z = (nn.mean() - r_exp) / (0.26136 / np.sqrt(n * density))
print(f"study area        : {hull_m.area/1e6:.0f} km^2")
print(f"median NN distance: {np.median(nn):.0f} m")
print(f"Clark-Evans R     : {R:.3f}  (R<1 => clustered; R=1 random)")
print(f"significance z    : {z:.1f}  (|z|>2 => significant)")

study area        : 996 km^2
median NN distance: 100 m
Clark-Evans R     : 0.542  (R<1 => clustered; R=1 random)
significance z    : -28.2  (|z|>2 => significant)


### The nearest-neighbour distribution

The single number behind the Clark-Evans test: how far is each plaque from its closest neighbour? A spike at very small distances is the signature of clustering.

In [24]:
fig_nn = px.histogram(x=nn, nbins=60, height=380,
                      labels={"x": "distance to nearest plaque (m)"},
                      title="How far is each plaque from its nearest neighbour?")
fig_nn.add_vline(x=float(np.median(nn)), line_dash="dash", line_color="#c2185b",
                 annotation_text=f"median {np.median(nn):.0f} m")
fig_nn.update_traces(marker_color="#1565c0")
fig_nn.show()

## Voronoi: the territory of your nearest plaque

Partition London into one cell per plaque: every point in a cell is closer to that plaque than any other. The cell *sizes* are the story: microscopic in the centre, enormous at the edges.

In [25]:
import shapely

mp = MultiPoint(list(map(tuple, X)))
envelope = hull_m.buffer(600, quad_segs=1)
cells = [g.intersection(envelope) for g in voronoi_diagram(mp, envelope=envelope).geoms]
cells_gdf = gpd.GeoDataFrame(geometry=cells, crs=27700)
cells_gdf["geometry"] = cells_gdf.simplify(120)          # metres: keep shape, cut vertices
cells_gdf["area_km2"] = cells_gdf.area / 1e6
cells_4326 = cells_gdf.to_crs(4326).reset_index(drop=True)
# Round coords to 5 dp (~1 m) so the embedded GeoJSON stays lean.
cells_4326["geometry"] = cells_4326.geometry.apply(
    lambda geom: shapely.transform(geom, lambda a: np.round(a, 5)))
cells_4326 = cells_4326[~cells_4326.geometry.is_empty]
cells_4326["log_area"] = np.log10(cells_4326["area_km2"].clip(lower=1e-3))

# Single continuous-colour trace (one shared GeoJSON) - colour = territory size.
fig_vor = px.choropleth_map(
    cells_4326, geojson=json.loads(cells_4326.to_json()), locations=cells_4326.index,
    color="log_area", color_continuous_scale="Blues_r", opacity=0.6,
    map_style="carto-positron", center=CENTER, zoom=9.6, height=640,
    title="Voronoi: the territory of your nearest blue plaque (darker = smaller)")
fig_vor.update_traces(marker_line_width=0.15)
fig_vor.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0}, coloraxis_showscale=False)
print(f"smallest territory {cells_4326['area_km2'].min():.3f} km^2 | "
      f"largest {cells_4326['area_km2'].max():.1f} km^2")
fig_vor.show()

smallest territory 0.000 km^2 | largest 31.1 km^2


## A density surface

The same clustering as a smooth heat map: a continuous view of where memory pools.

In [26]:
fig_den = px.density_map(geo, lat="lat", lon="lng", radius=14, center=CENTER,
                         zoom=9.6, height=620, color_continuous_scale="Turbo",
                         title="Density of London's blue plaques")
fig_den.update_layout(map_style="carto-darkmatter", margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig_den.show()

## Hotspots by algorithm (DBSCAN)

Let a density-based clustering algorithm find the hotspots itself: any group of ≥ 6 plaques within 350 m of each other becomes a cluster; the rest are “scattered”.

In [27]:
rad = np.radians(geo[["lat", "lng"]].values)
db = DBSCAN(eps=350 / 6_371_000, min_samples=6, metric="haversine").fit(rad)
geo["cluster"] = db.labels_
n_clusters = len({l for l in db.labels_ if l != -1})
hotspots = (geo[geo["cluster"] != -1].groupby("cluster")
            .agg(size=("name", "size"), borough=("borough_short", lambda s: s.mode().iat[0]))
            .sort_values("size", ascending=False))
print(f"{n_clusters} hotspots (eps=350 m, min_samples=6)")
print(hotspots.head(6).to_string())

plot = geo.copy()
plot["hotspot"] = np.where(plot["cluster"] == -1, "scattered", "in a hotspot")
fig_db = px.scatter_map(plot, lat="lat", lon="lng", color="hotspot", hover_name="name",
                        center=CENTER, zoom=9.6, height=640,
                        color_discrete_map={"scattered": "#bbbbbb", "in a hotspot": "#1565c0"},
                        title=f"DBSCAN finds {n_clusters} plaque hotspots")
fig_db.update_layout(map_style="carto-positron", margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig_db.show()

14 hotspots (eps=350 m, min_samples=6)
         size                 borough
cluster                              
0         287             Westminster
1         232  Kensington And Chelsea
2          42                  Camden
5          19                  Camden
4          13             Westminster
8          13             Westminster


## The memory network (minimum spanning tree)

The shortest possible set of links connecting **every** plaque into one network: a graph-theory portrait of the whole scheme.

In [28]:
D = squareform(pdist(X))
mst = minimum_spanning_tree(D).tocoo()
total_km = mst.sum() / 1000
lat_ll, lon_ll = pts.geometry.y.values, pts.geometry.x.values
edge_lat, edge_lon = [], []
for i, j in zip(mst.row, mst.col):
    edge_lat += [lat_ll[i], lat_ll[j], None]
    edge_lon += [lon_ll[i], lon_ll[j], None]
fig_mst = go.Figure(go.Scattermap(lat=edge_lat, lon=edge_lon, mode="lines",
                                  line={"width": 1, "color": "#1565c0"}, hoverinfo="skip"))
fig_mst.add_trace(go.Scattermap(lat=lat_ll, lon=lon_ll, mode="markers",
                                marker={"size": 3, "color": "#c2185b"}, hoverinfo="skip"))
fig_mst.update_layout(map_style="carto-positron", map_center=CENTER, map_zoom=9.6, height=640,
                      showlegend=False, margin={"r": 0, "t": 40, "l": 0, "b": 0},
                      title=f"Memory network: MST linking all {n} plaques ({total_km:.0f} km)")
print(f"minimum spanning tree length: {total_km:.0f} km")
fig_mst.show()

minimum spanning tree length: 371 km


## The Grand Tour of London (Travelling Salesman)

Pick one representative plaque per borough and find the shortest loop that visits them all. Solved from scratch: a **nearest-neighbour** first guess, then **2-opt** uncrossing until no swap helps.

In [29]:
# One representative plaque per borough: the one nearest its borough centroid.
reps = []
for b, sub in pts_m.groupby("borough_short"):
    c = sub.geometry.union_all().centroid
    d = (sub.geometry.x - c.x) ** 2 + (sub.geometry.y - c.y) ** 2
    reps.append(sub.loc[d.idxmin()])
rep = gpd.GeoDataFrame(reps, crs=27700)
P = np.c_[rep.geometry.x.values, rep.geometry.y.values]
DM = squareform(pdist(P))

def tour_len(t):
    return sum(DM[t[k], t[(k + 1) % len(t)]] for k in range(len(t)))

def nearest_neighbour(dist):
    unv, t = set(range(1, len(dist))), [0]
    while unv:
        nxt = min(unv, key=lambda j: dist[t[-1], j])
        t.append(nxt); unv.discard(nxt)
    return t

def two_opt(t):
    best, improved = t[:], True
    while improved:
        improved = False
        for a in range(1, len(best) - 1):
            for c in range(a + 1, len(best)):
                if c - a == 1:
                    continue
                cand = best[:a] + best[a:c][::-1] + best[c:]
                if tour_len(cand) < tour_len(best):
                    best, improved = cand, True
    return best

t_nn = nearest_neighbour(DM); nn_len = tour_len(t_nn) / 1000
t_opt = two_opt(t_nn); opt_len = tour_len(t_opt) / 1000
print(f"nearest-neighbour tour: {nn_len:.1f} km")
print(f"after 2-opt          : {opt_len:.1f} km  ({(1-opt_len/nn_len)*100:.1f}% shorter)")

rep_ll = rep.to_crs(4326)
lat_r, lon_r = rep_ll.geometry.y.values, rep_ll.geometry.x.values
order = t_opt + [t_opt[0]]
fig_tsp = go.Figure(go.Scattermap(
    lat=[lat_r[k] for k in order], lon=[lon_r[k] for k in order], mode="lines+markers",
    line={"width": 2, "color": "#1565c0"}, marker={"size": 8, "color": "#c2185b"},
    text=[rep_ll.iloc[k]["borough_short"] for k in order], hoverinfo="text"))
fig_tsp.update_layout(map_style="carto-positron", map_center=CENTER, map_zoom=9.4, height=640,
                      margin={"r": 0, "t": 40, "l": 0, "b": 0},
                      title=f"The Grand Tour: shortest loop through {len(t_opt)} boroughs ({opt_len:.0f} km)")
fig_tsp.show()

nearest-neighbour tour: 219.0 km
after 2-opt          : 179.5 km  (18.0% shorter)


## Takeaways

- London's plaques are **statistically, overwhelmingly clustered** (Clark-Evans R ≈ 0.54, z ~ -28).
- The **Voronoi cells** make it visceral: central plaques own tiny patches, outer-London plaques own boroughs.
- **DBSCAN** recovers the obvious hotspots (Westminster, Kensington, Bloomsbury) without being told where to look.
- A hand-rolled **nearest-neighbour + 2-opt** TSP shaves ~18% off the naive route, the classic value of local search.

In [30]:
EXP = Path("exports"); EXP.mkdir(exist_ok=True)

VIEWPORT_META = '<meta name="viewport" content="width=device-width, initial-scale=1" />'


def inject_viewport(path: Path) -> None:
    """Plotly's write_html omits a viewport meta; add one so embeds render
    at device width inside iframes on phones/tablets."""
    html = path.read_text()
    if 'name="viewport"' in html:
        return
    html = html.replace('<meta charset="utf-8" />', '<meta charset="utf-8" />' + VIEWPORT_META, 1)
    path.write_text(html)


html_config = {"responsive": True, "displaylogo": False}
for name, fig in [("voronoi", fig_vor), ("density", fig_den), ("hotspots", fig_db),
                  ("memory-network", fig_mst), ("grand-tour", fig_tsp), ("nn-distances", fig_nn)]:
    out = EXP / f"{name}.html"
    fig.write_html(out, include_plotlyjs="cdn", config=html_config)
    inject_viewport(out)
print("exports written:", sorted(p.name for p in EXP.glob("*.html")))


exports written: ['by-borough.html', 'by-category.html', 'density.html', 'female-by-field.html', 'gender-by-decade.html', 'grand-tour.html', 'honorifics.html', 'hotspots.html', 'inscription-verbs.html', 'inscription-words.html', 'lifespans.html', 'materials.html', 'memory-network.html', 'nn-distances.html', 'plaque-map.html', 'postcode-density.html', 'top-streets.html', 'voronoi.html']
